In [ ]:
import warnings 
warnings.filterwarnings("ignore", message=r"Passing", category=FutureWarning)
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/Engram_2P/Engram_2P/utilities',
                 '/Users/amonast/Documents/GitHub/Engram_2P/Engram_2P/rois'])
from traces import plot_deconv,ev_trace
from rois import remove_bad_cells
import matplotlib.pyplot as plt
import holoviews as hv
import numpy as np
import os
import pandas as pd
hv.extension('bokeh')
import pickle5

In [4]:
def get_traces(animal, fov, session, file_key,base_dir):
    metadata = pd.read_csv(file_key)
    deconv_path = os.path.join(base_dir, 'deconvolution','deconvolution_results','deconv_results_min5')
    TSeries = metadata['TSeries_g'].loc[(metadata['Animal'] == animal) & (metadata['FOV'] == fov) & (metadata['Session'] == session)].values[0]
    dfile = [os.path.join(deconv_path, f) for f in os.listdir(deconv_path) if ((TSeries in f)&(f.endswith('l0deconv.pkl')))][0]
    print('deconvolution file: '+ dfile)
    with open(dfile, 'rb') as file:
        data = pickle5.load(file)
    dff = data['dff']
    times = data['times']
    ev = data['events']
    est = data['est']

    return dff, ev, times, est

In [ ]:
file_key,base_dir = '/Volumes/AM_SSD3/Tone2P/Data_info_TFC.csv','/Volumes/AM_SSD3/Tone2P'
ani,fov ='194L','FOV1'
dff,ev,times,est = get_traces(ani,fov,'Recall1',file_key,base_dir)
inds = remove_bad_cells(ani, fov, snr_thr=4.0,file_key=file_key,base_dir=base_dir)
recall_inds = inds['Recall1']

In [ ]:
mousey = animal.animal(animal=ani,fov='FOV1',file_key=file_key,base_dir=base_dir)
traces_df = mousey.load_traces(sessions=['Recall1'],split=False)

Check traces quickly, note any abnormal

In [ ]:
from traces import ev_trace
dec = ev_trace(ev,times,dff.shape[1])
for i in recall_inds:
    
    plt.figure(figsize=(20,3))
    plt.plot(dff[i,:])
    plt.plot(dec[i,:])
    plt.title('Baseline roi '+str(i))
    plt.show()
    # key = input('press key')
    # if key=='q':
    #     break
    plt.close()











interactive plot for cells that need more close zoom

In [6]:
D = dict({'dff':dff,'estimated_calcium':est,'spiketimes':times,'event_mag':ev})

In [ ]:
import numpy.random as random 
N=20
random_i  = random.choice(np.arange(0, dff.shape[0]), size=N, replace=False)
curve_dict_1 = {ii: plot_deconv(D,i,dff.shape[1],frame_rate=30) for ii,i in enumerate(random_i)}
plot = hv.HoloMap(curve_dict_1,kdims='roi') 
plot

In [17]:
import holoviews as hv

In [ ]:
import panel as pn
from bokeh.resources import INLINE

pn.pane.HoloViews(plot).save('/Users/amonast/Desktop/Caiman/Analysis/rates/decmin5_034R_baseline_cells.html',embed=True, resources=INLINE)


In [ ]:
curve_dict_1 = {i: plot_deconv(D,i,dff.shape[1],frame_rate=30) for ii,i in enumerate(np.arange(0,100,2))}
plot = hv.HoloMap(curve_dict_1,kdims='roi') 
plot

In [ ]:
import panel as pn
from bokeh.resources import INLINE
pn.pane.HoloViews(plot).save('/Users/amonast/Desktop/Caiman/Analysis/rates/decmin4_034R_baseline_cells_more.html',embed=True, resources=INLINE)


In [ ]:
dff,ev,times,est = get_traces(ani,fov,'Post',file_key,base_dir)
post = inds.Post.values[inds.Post.values!=-1]
dec = ev_trace(ev,times,dff.shape[1])
for i in post:
    plt.figure(figsize=(20,3))
    plt.plot(dff[i,:])
    plt.plot(dec[i,:])
    plt.title('Post roi '+str(i))
    plt.show()
    # key = input('press key')
    # if key=='q':
    #     break
    plt.close()

    
    

In [ ]:
D = dict({'dff':dff,'estimated_calcium':est,'spiketimes':times,'event_mag':ev})
curve_dict_1 = {ii: plot_deconv(D,i,dff.shape[1],frame_rate=30) for ii,i in enumerate([249])}
plot = hv.HoloMap(curve_dict_1,kdims='roi') 
plot